# 26. The 2026-27 season runner (simulator step 3)

Notebooks 23-25 built the pieces: lineups and minutes, a game model, and a team bonus. This notebook puts them together on the **real 2026-27 schedule** (84 games per team), so we can ask "what if" questions:

- **From opening night,** or **from any later date**. Real results before that date are locked in, and only the rest of the season is simulated.
- **With or without trades:** the real ones in `data/trades_2026-27.csv`, ticked on or off.
- **With players sitting out:** Larkin and Edvinsson in Detroit, for example.

**The output** for every team, from 10,000 simulated seasons:

- points (average and a range);
- wins, losses, OT losses and goal difference;
- playoff odds, division title odds, Presidents' Trophy odds, and average final rank.

**What runs it** (code: `src/season.py`): notebook 23's lineup rules with notebook 24's settings (call-up tiers, minutes blend 0.75), notebook 24's game model with **all-seasons coefficients**, notebook 17's player randomness, and notebook 25's **team bonus** (the residual, lambda 0.4). That's the step 3 default Ryan chose in notebook 25.

**What's new here:**

- the real schedule;
- real results locked in;
- a goalie rule for back-to-backs;
- the NHL's playoff format;
- a saved "bundle" of inputs, so the Simulator page can run scenarios without recomputing anything.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import time
import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, project, backtest, aging_curve, group_average, current_projections, add_age, GOALS_PER_WIN
from edge_data import (build_bio_table, schedule_games, nhl_code, download_season_schedule, season_schedule,
                       download_standings, standings_detail)
from nhle import load_profiles
from development import (training_nhle, best_nhle, skater_features, fit_prior, apply_prior, candidates, fit_rows,
                         experience, TARGETS, AGING, SK_WEIGHTS, SK_REG, G_WEIGHTS, G_REG, PRIOR_INPUTS)
from ratings import ovr
import lineups as L
import simulate as S
import season as SE

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
REFRESH = False          # True: download the 2026-27 schedules again to pick up new results (33 NHL calls)
BUNDLE_DIR = DATA / "simulator_2026-27"
TRADES_PATH = DATA / "trades_2026-27.csv"

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]
SCORE_TRAIN = [t for t in TRAIN if t >= 2013]          # notebook 25's scored training seasons (for beta)
CHOSEN = {"F": (0.75, "R1"), "D": (0.75, "R1"), "G": (0.5, "R1")}    # notebook 23: coach score
LOOKBACK = {"F": 2, "D": 1, "G": 2}                                  # notebook 23: seasons of usage
N_SIMS = 1000                                                        # notebook 25's simulations (settings rebuild)
N_RUN, SEED = 10000, 26                                              # this notebook's scenarios

## 2. The 2026-27 schedule, results so far, and divisions

- **Schedules:** the NHL API's season schedule for each of the 32 teams, saved to `data/schedule_raw/2026-27/` (gitignored).
  - The files sit in a subfolder because notebooks 24 and 25 read every file in `data/schedule_raw/`. In the same folder, this season's games would change their results.
  - With `REFRESH = False`, saved files are reused and **nothing is downloaded**. Set it to True to pull new results.
- **Today** is the day the schedules were downloaded (stamped in the files), not the computer's clock. So a rerun gives the same numbers.
- **Divisions and conferences** come from the NHL standings for today (one call, saved once). Those standings also **check the results locked in:** points, wins, OT losses, regulation wins and regulation + OT wins, team by team.

In [2]:
rosters = pd.read_csv(DATA / "rosters_2026-27.csv")
TEAMS = sorted(rosters["team_opening_night"].dropna().unique())
failed = download_season_schedule(TEAMS, DATA / "schedule_raw" / "2026-27", refresh=REFRESH)
assert not failed
SCHED, STAMP = season_schedule(DATA / "schedule_raw" / "2026-27")
TODAY = STAMP[:10]
gp = pd.concat([SCHED["home"], SCHED["away"]]).value_counts()
assert len(SCHED) == 1344 and (gp == SE.GAMES_PER_TEAM).all() and len(gp) == 32
assert SCHED["date"].min() == SE.OPENING_NIGHT
played = SCHED[SCHED["played"]]
assert played["date"].max() < TODAY                     # every result is from before today
print(f"Schedule: {len(SCHED):,} games, {SE.GAMES_PER_TEAM} per team, {SCHED['date'].min()} to {SCHED['date'].max()}")
print(f"Downloaded {STAMP}: {len(played)} games played through {played['date'].max()} "
      f"(REG {(played['last_period'] == 'REG').sum()}, OT {(played['last_period'] == 'OT').sum()}, SO {(played['last_period'] == 'SO').sum()})")

DIV = standings_detail(download_standings(TODAY, DATA / "current_raw" / f"standings_{TODAY}.json"))
mine = SE.real_so_far(SCHED, TODAY)
theirs = DIV.set_index("team")[["gp", "wins", "otl", "points", "rw", "row"]]
assert (mine[theirs.columns].sort_index() == theirs.sort_index()).all().all()
print(f"\nResults so far match the NHL standings for {TODAY} for all 32 teams (points, W, OTL, RW, ROW)")
for (conf, div), d in DIV.groupby(["conference", "division"]):
    print(f"  {conf:8s} {div:13s}: {', '.join(d['team'])}")

32 already saved, 0 downloaded
Schedule: 1,344 games, 84 per team, 2026-09-29 to 2027-04-10
Downloaded 2026-11-10 11:13: 285 games played through 2026-11-09 (REG 222, OT 43, SO 20)

Results so far match the NHL standings for 2026-11-10 for all 32 teams (points, W, OTL, RW, ROW)
  Eastern  Atlantic     : BOS, BUF, DET, FLA, MTL, OTT, TBL, TOR
  Eastern  Metropolitan : CAR, CBJ, NJD, NYI, NYR, PHI, PIT, WSH
  Western  Central      : CHI, COL, DAL, MIN, NSH, STL, UTA, WPG
  Western  Pacific      : ANA, CGY, EDM, LAK, SEA, SJS, VAN, VGK


## 3. The settings, rebuilt from notebook 25 and checked

The runner needs notebook 25's step 3 default:

- **the game model** fit on all seasons;
- **beta:** points per goal of edge;
- **every team's residual** from 2025-26.

These came out of notebook 25's backtest pipeline, so this section reruns its **all-seasons path**: past projections (notebook 18, with its error check), pools of players with 1+ games, lineups, the fit, then 1,000 simulations of 2025-26.

The asserts check it reproduces notebook 25's printed numbers.

In [3]:
skater_war = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalie_war = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
history = history_table(skater_war, goalie_war)
skaters, goalies = history[history["group"] != "G"], history[history["group"] == "G"]
bios = build_bio_table(DATA / "bio_raw", 2026)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")

old = schedule_games(DATA / "schedule_raw")
recent = S.games_from_shots(pd.read_parquet(DATA / "shots_2021-2025.parquet"))
games = pd.concat([old, recent], ignore_index=True)
results = S.add_results(games)
results["reg_goals"] = results["home_goals"] + results["away_goals"] - results["ot"]
real = S.standings(games)
LATE = S.late_table(recent)
reg_pg = results.groupby("season")["reg_goals"].mean()

def normal_total(seasons):
    """Normal goals per game going into a season: regulation goals per game in `seasons` minus the late goals."""
    return float(reg_pg.reindex(seasons).dropna().mean() - LATE["extra_goals"])

f_train, m_train = training_nhle(careers, TRAIN_END)
nhle_train = best_nhle(careers, f_train, m_train, rule="last2")
pairs = pd.concat([candidates(history, TARGETS + [2026]),
                   skaters[skaters["season"].isin(TARGETS)][["playerId", "season", "group"]]]).drop_duplicates(["playerId", "season"])
feat = skater_features(pairs, history, bios, profiles, nhle_train)
model = fit_prior(fit_rows(history, feat, TRAIN), PRIOR_INPUTS)
cand = candidates(history, TARGETS).merge(feat.drop(columns="group"), on=["playerId", "season"], how="left")
priors = cand[["playerId", "season"]].assign(prior_gar_60=apply_prior(cand, model).to_numpy())
rows = []
for t in TARGETS:
    curve = aging_curve(history, bios, exclude_season=t)[1]
    ps = project(skaters, bios, t, SK_WEIGHTS, SK_REG, AGING, curve, priors=priors, prior_reg=1500)
    pg = project(goalies, bios, t, G_WEIGHTS, G_REG, AGING, curve)
    rows.append(pd.concat([ps, pg])[["proj_war_60"]].assign(season=t).rename_axis("playerId").reset_index())
bt_proj = pd.concat(rows, ignore_index=True)
b = backtest(skaters, TEST, SK_WEIGHTS, SK_REG, 300, bios, aging=AGING, priors=priors, prior_reg=1500)
b = b.merge(feat[["playerId", "season", "age"]], on=["playerId", "season"], how="left")
k = b["age"] > 0
err = np.sqrt(np.average((b["actual"][k] - b["pred"][k]) ** 2, weights=b["toi"][k]))
assert round(err, 4) == 0.2665, err
print(f"Past-season projections rebuilt; test-season error {err:.4f} = notebook 18's")

Past-season projections rebuilt; test-season error 0.2665 = notebook 18's


In [4]:
faceoffs = L.faceoff_rates([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"])
usage = L.usage_table(skater_war, goalie_war, faceoffs)
exp = experience(history, profiles, history[["playerId", "season"]])

def make_pool(t):
    """Players with 1+ games for a team in season t, with projection and usage (notebook 24's main pool)."""
    u = usage[(usage["season"] == t) & (usage["games_played"] >= 1)]
    p = (u[["playerId", "name", "team", "position", "group", "games_played", "usage"]]
         .rename(columns={"games_played": "gp", "usage": "actual"}).reset_index(drop=True))
    p["proj_war_60"] = p["playerId"].map(bt_proj[bt_proj["season"] == t].set_index("playerId")["proj_war_60"])
    miss = p["proj_war_60"].isna() & (p["group"] != "G")
    f = p.loc[miss, ["playerId", "group"]].merge(feat[feat["season"] == t].drop(columns="group"), on="playerId", how="left")
    p.loc[miss, "proj_war_60"] = (apply_prior(f, model) / GOALS_PER_WIN).to_numpy()
    still = p["proj_war_60"].isna()
    p.loc[still, "proj_war_60"] = p.loc[still, "group"].map(group_average(history, t - 1) / GOALS_PER_WIN)
    p["usage_last"] = L.usage_by_group(usage, p, t, LOOKBACK)
    p["natural_center"] = L.natural_center(usage, p["playerId"], p["position"], t) & (p["group"] == "F")
    g = p["group"] == "G"
    p.loc[g, "actual"] = p.loc[g, "gp"] / p[g].groupby("team")["gp"].transform("sum")
    p["season"] = t
    return p

lineups = {}
for t in TARGETS:
    seasons = [t - 3, t - 2, t - 1]
    defaults, slots = L.rookie_defaults(usage, exp, seasons), L.slot_minutes(usage, seasons)
    share, totals = L.starter_share(usage, seasons)[1], L.team_minutes(usage, seasons)
    sc = L.coach_scores(make_pool(t), CHOSEN, defaults)
    lu = pd.concat([L.pick_lineup(x) for _, x in sc.groupby("team")])
    lu["team"] = [nhl_code(x, t) for x in lu["team"]]
    lu["minutes_per_game"] = L.assign_minutes(lu, L.scaled_slots(slots, totals), share, own_weight=SE.OWN_WEIGHT)
    lu["age"] = add_age(lu[["playerId"]].assign(season=t), bios)["age"].to_numpy()
    lu["draft_overall"] = lu["playerId"].map(bios.set_index("playerId")["draft_overall"])
    lineups[t] = lu.reset_index(drop=True)

rows = []
for t in TARGETS:
    teams, sk, gk = S.team_strength(lineups[t])
    rows.append(pd.DataFrame({"season": t, "team": teams, "skater": sk[0], "goalie": gk[0]}))
STRENGTH = pd.concat(rows, ignore_index=True)
s = STRENGTH.set_index(["season", "team"])
GAMES = results[results["season"].isin(TARGETS)].copy()
h = s.reindex(pd.MultiIndex.from_arrays([GAMES["season"], GAMES["home"]]))
a = s.reindex(pd.MultiIndex.from_arrays([GAMES["season"], GAMES["away"]]))
assert h.notna().all().all() and a.notna().all().all()
GAMES["ds"] = h["skater"].to_numpy() - a["skater"].to_numpy()
GAMES["dg"] = h["goalie"].to_numpy() - a["goalie"].to_numpy()
print(f"Past lineups for {len(TARGETS)} seasons, {len(GAMES):,} games with both teams' strength")

Past lineups for 14 seasons, 17,711 games with both teams' strength


In [5]:
INPUTS = S.draw_inputs(pd.read_csv(DATA / "dev_residuals.csv"), pd.read_csv(DATA / "dev_ranges.csv"), center_pools=True)
REAL = real[["season", "team", "gp", "points"]].rename(columns={"points": "real"})
MOVED = {("ATL", 2011): "WPG", ("PHX", 2014): "ARI", ("ARI", 2024): "UTA"}

def fit_all(t):
    """Notebook 25's "all seasons" fit for target season t: every target season before it, equal weights."""
    seasons = [s_ for s_ in TARGETS if s_ < t]
    return S.fit_game_model(GAMES[GAMES["season"].isin(seasons)], LATE, normal_total(seasons))

def simulate_past(fit, season):
    """Notebook 25's simulate(): 1,000 seasons of a past season, its real schedule, its seed."""
    lu = lineups[season]
    sched = results[results["season"] == season]
    gp_ = pd.concat([sched["home"], sched["away"]]).value_counts()
    buckets = S.draw_buckets(lu["group"], lu["age"], lu["draft_overall"], INPUTS)
    draws = S.player_draws(lu["group"].to_numpy(), buckets, lu["minutes_per_game"] * lu["team"].map(gp_).to_numpy(),
                           INPUTS, "full", N_SIMS, seed=season)
    sett = S.game_settings(fit, LATE, normal_total([season - 3, season - 2, season - 1]))
    table, _ = S.simulate_season(lu, sched[["home", "away"]], sett, N_SIMS, seed=season, draws=draws)
    return table.assign(season=season)

# beta: points per game for one normal goal per game of edge (notebook 25, training seasons)
d = pd.concat([simulate_past(fit_all(t), t) for t in SCORE_TRAIN]).merge(REAL, on=["season", "team"]).merge(STRENGTH, on=["season", "team"])
f = {t: fit_all(t) for t in SCORE_TRAIN}
d["q"] = [f[t]["a_s"] * s_ + f[t]["a_g"] * g_ for t, s_, g_ in zip(d["season"], d["skater"], d["goalie"])]
d["q"] -= d.groupby("season")["q"].transform("mean")
BETA = float(np.polyfit(d["q"], d["points"] / d["gp"], 1)[0])

# The 2026-27 game model (all seasons) and each team's residual from 2025-26
FIT26 = fit_all(2026)
x = simulate_past(FIT26, 2025).merge(REAL, on=["season", "team"])
r = (x["real"] - x["points"]) / x["gp"]
RESID = pd.Series((r - r.mean()).to_numpy(), index=x["team"]).rename(index=lambda tm: MOVED.get((tm, 2026), tm))
BONUS = (SE.TEAM_LAMBDA * RESID / BETA).to_dict()
RECENT = [2023, 2024, 2025]
GAME26 = S.game_settings(FIT26, LATE, normal_total(RECENT))

checks = {"a_s": (FIT26["a_s"], 5.925), "a_g": (FIT26["a_g"], 8.427), "home": (FIT26["home"], 0.234),
          "beta": (BETA, 0.379), "bonus DET": (BONUS["DET"], 0.103), "bonus VAN": (BONUS["VAN"], -0.422),
          "bonus COL": (BONUS["COL"], 0.320)}
for name, (mine_, nb25) in checks.items():
    assert round(mine_, 3) == nb25, (name, mine_)
print("Notebook 25's step 3 default reproduced: " + ", ".join(f"{k} {v[0]:.3f}" for k, v in checks.items()))
print(f"Normal scoring for 2026-27: {GAME26['total']:.3f} goals per game (2023-24 to 2025-26)")

Notebook 25's step 3 default reproduced: a_s 5.925, a_g 8.427, home 0.234, beta 0.379, bonus DET 0.103, bonus VAN -0.422, bonus COL 0.320
Normal scoring for 2026-27: 5.456 goals per game (2023-24 to 2025-26)


## 4. The 2026-27 players, and the saved bundle

- **Players:** every 2026-27 player with his opening-night team and status (notebook 22), projection (notebooks 18-22), and coach score (notebook 23), exactly as in notebooks 24 and 25. The OVR shown on the Forecast page is attached for display.
- **Overtime vs. shootout:** the game model only says a game went past regulation. The regulation + OT wins tiebreaker needs to know which games were shootouts, so each simulated OT/SO game is a shootout with the real share (2021-22 to 2025-26). That never changes points.
- **The bundle:** `data/simulator_2026-27/` (gitignored) holds everything a run needs. A run never recomputes WAR and never calls the NHL. The check below rebuilds every lineup from the saved files and compares it with the one in memory.

In [6]:
players = L.load_players(DATA, usage, current_projections(DATA), LOOKBACK)
DEFAULTS = L.rookie_defaults(usage, exp, RECENT)
SLOTS26 = L.scaled_slots(L.slot_minutes(usage, RECENT), L.team_minutes(usage, RECENT))
SHARE = L.starter_share(usage, RECENT)[1]
scored26 = L.coach_scores(players[players["in_pool"]], CHOSEN, DEFAULTS)
scored26 = scored26.merge(current_projections(DATA)[["playerId", "age"]], on="playerId", how="left")
scored26["draft_overall"] = scored26["playerId"].map(bios.set_index("playerId")["draft_overall"])
ft = pd.read_csv(DATA / "forecast_table_2026-27.csv").set_index("playerId")
scored26["ovr"] = scored26["playerId"].map(ft["ovr"])
miss = scored26["ovr"].isna()
scored26.loc[miss, "ovr"] = ovr(scored26.loc[miss, "proj_war_60"], scored26.loc[miss, "group"]).astype(float)
print(f"2026-27 pool players: {len(scored26):,}; OVR from the Forecast table for {(~miss).sum():,}, computed for {miss.sum()}")

ot_games = recent[recent["last_period"] != "REG"]
SO_SHARE = float((ot_games["last_period"] == "SO").mean())
print(f"Shootouts: {SO_SHARE:.1%} of games that went past regulation (2021-22 to 2025-26)")

SE.save_bundle(BUNDLE_DIR, scored26, SCHED, STAMP, GAME26, BONUS, SLOTS26, SHARE, DIV, SO_SHARE, INPUTS)
B = SE.load_bundle(BUNDLE_DIR, TRADES_PATH)
mem = L.build_lineups(scored26, SLOTS26, SHARE, tiered=SE.TIERED, own_weight=SE.OWN_WEIGHT)
disk = SE.scenario_lineups(B)
cols = ["playerId", "team", "dressed", "rank", "slot", "proj_war_60"]
assert mem[cols].reset_index(drop=True).equals(disk[cols].reset_index(drop=True))
# minutes: the same up to the last digit a number can lose when it's written to a text file and read back
assert np.allclose(mem["minutes_per_game"], disk["minutes_per_game"], rtol=0, atol=1e-9)
print(f"Bundle saved to {BUNDLE_DIR} and read back: all 32 lineups identical "
      f"({disk['dressed'].sum()} dressed players)")
print("Trades on file:", ", ".join(f"{t} ({d})" for t, d in B["trades"].drop_duplicates("trade_id")[["trade_id", "date"]].values))

2026-27 pool players: 1,348; OVR from the Forecast table for 1,348, computed for 0
Shootouts: 33.0% of games that went past regulation (2021-22 to 2025-26)
Bundle saved to ../data/simulator_2026-27 and read back: all 32 lineups identical (640 dressed players)
Trades on file: T1 (2026-10-23), T2 (2026-10-10), T3 (2026-10-10), T4 (2026-10-25), T5 (2026-10-28), T6 (2026-11-03)


## 5. Who starts in goal

**The rule:**

- The **backup starts the second night of every back-to-back**, from the real dates.
- A starter's share is about 0.643 of games (notebook 23, from 2023-24 to 2025-26), so a backup gets round(0.357 x 84) = **30 starts**. Back-to-backs give some of them. The rest are **spread evenly** over the team's other games.
- **It's the same every run:** no randomness, and real NHL coaches plan it that way too.
- **If a team ever had more than 30 back-to-backs,** the backup would get just those.

**In a game,** each goalie counts at his projected WAR per 60 (plus his random draw) for the full 60 minutes. Over a season that averages out to the same starter/backup mix as notebooks 24 and 25. The difference is *which* games the backup plays: tired nights, against whoever is on the schedule.

In [7]:
gs = B["goalie_starts"].attrs["summary"]
print(f"Back-to-backs per team: {gs['back_to_backs'].min()} to {gs['back_to_backs'].max()} (median {gs['back_to_backs'].median():.0f}); "
      f"backup starts {gs['backup_starts'].min()} to {gs['backup_starts'].max()}; "
      f"starter's share {gs['starter_share'].min():.3f} to {gs['starter_share'].max():.3f}")
over = gs[gs["back_to_backs"] > gs["backup_starts"].min()]
print("Teams with more back-to-backs than the 30-start target:", ", ".join(over["team"]) if len(over) else "none")
print(gs.sort_values("back_to_backs", ascending=False).head(5).to_string(index=False))

Back-to-backs per team: 8 to 15 (median 12); backup starts 30 to 30; starter's share 0.643 to 0.643
Teams with more back-to-backs than the 30-start target: none
team  back_to_backs  backup_starts  starter_share
 PIT             15             30       0.642857
 WSH             14             30       0.642857
 FLA             14             30       0.642857
 BOS             14             30       0.642857
 OTT             14             30       0.642857


## 6. The opening-night baseline

Everything from opening night: **no trades** (they all happened later), **the disputes play** (Larkin and Edvinsson dress, as in notebook 23), and the team bonus on. 10,000 simulated seasons.

**Standings rules:**

- **Playoffs:** the top 3 in each division, plus the 2 best other teams in each conference (wild cards).
- **Ties are broken by** regulation wins, then regulation + overtime wins, then a coin flip.
- **Real tiebreakers skipped** (said plainly): total wins, points in head-to-head games, goal difference, goals for. They rarely matter after the first two.

In [8]:
def by_division(table, cols=None):
    """Standings, one block per division, best first."""
    cols = cols or ["team", "points", "p10", "p90", "wins", "losses", "otl", "goal_diff", "playoff_pct", "division_pct",
                    "presidents_pct", "avg_rank"]
    for (conf, div), d in table.groupby(["conference", "division"]):
        print(f"{conf}, {div}")
        print(d.sort_values("points", ascending=False)[cols].round(1).to_string(index=False))
        print()

BASE = SE.run_scenario(B, n_sims=N_RUN, seed=SEED)
print(f"{BASE['n_sims']:,} seasons from {BASE['start_date']} ({BASE['games_simulated']:,} games simulated, "
      f"{BASE['games_locked']} locked) in {BASE['seconds']:.1f} seconds; trades applied: {BASE['trades'] or 'none'}\n")
by_division(BASE["table"])

10,000 seasons from 2026-09-29 (1,344 games simulated, 0 locked) in 5.6 seconds; trades applied: none

Eastern, Atlantic
team  points  p10   p90  wins  losses  otl  goal_diff  playoff_pct  division_pct  presidents_pct  avg_rank
 BUF   104.0 87.0 121.0  47.9    28.0  8.1       38.0         75.4          26.5             8.0      10.2
 MTL   102.8 85.0 120.0  47.3    28.5  8.2       33.6         72.9          23.8             6.9      10.8
 TBL   101.1 83.0 118.0  46.4    29.3  8.4       28.1         68.2          20.4             5.7      11.7
 BOS    94.8 77.0 113.0  43.1    32.3  8.7        6.8         50.6          10.0             2.4      15.4
 DET    91.2 74.0 109.0  41.1    33.9  9.0       -5.4         40.2           6.0             1.0      17.5
 FLA    89.8 72.0 108.0  40.4    34.6  9.0      -10.6         36.0           5.3             0.8      18.3
 OTT    89.0 71.0 107.0  40.0    35.0  9.0      -13.0         33.7           5.2             1.1      18.8
 TOR    85.1 67.0 103.0

## 7. Detroit: the two disputes

Four versions from opening night, same seed: (a) as is, (b) Larkin sits, (c) Edvinsson sits, (d) both sit.

Each player has **the same random draw in every scenario** (draws are made once for the whole league, in a fixed order). So a difference comes from the roster change, not from new random numbers. Someone else dresses in each sitting player's place, the next player up by notebook 23's coach score.

In [9]:
pid = lambda name: int(B["players"].loc[B["players"]["name"] == name, "playerId"].iloc[0])
LARKIN, EDVINSSON = pid("Dylan Larkin"), pid("Simon Edvinsson")

def team_row(res, team):
    x = res["table"].set_index("team").loc[team]
    return {"points": x["points"], "10th": x["p10"], "90th": x["p90"], "playoff %": x["playoff_pct"]}

DET = {"(a) as is": BASE}
for name, out in [("(b) Larkin sits", [LARKIN]), ("(c) Edvinsson sits", [EDVINSSON]), ("(d) both sit", [LARKIN, EDVINSSON])]:
    DET[name] = SE.run_scenario(B, sit_out=out, n_sims=N_RUN, seed=SEED)
det = pd.DataFrame({k: team_row(v, "DET") for k, v in DET.items()}).T
det["points vs. (a)"] = det["points"] - det.loc["(a) as is", "points"]
det["playoff % vs. (a)"] = det["playoff %"] - det.loc["(a) as is", "playoff %"]
print("Detroit, 2026-27 from opening night:")
print(det.round(2).to_string())
for name in ["(b) Larkin sits", "(c) Edvinsson sits"]:
    lu, base_lu = DET[name]["lineups"], BASE["lineups"]
    a_ = base_lu[(base_lu["team"] == "DET") & base_lu["dressed"]]["name"]
    b_ = lu[(lu["team"] == "DET") & lu["dressed"]]["name"]
    print(f"{name}: in {', '.join(sorted(set(b_) - set(a_)))} for {', '.join(sorted(set(a_) - set(b_)))}")

Detroit, 2026-27 from opening night:
                    points  10th   90th  playoff %  points vs. (a)  playoff % vs. (a)
(a) as is            91.21  74.0  109.0      40.21            0.00               0.00
(b) Larkin sits      87.71  70.0  106.0      31.19           -3.49              -9.02
(c) Edvinsson sits   90.43  72.0  108.0      38.22           -0.77              -1.99
(d) both sit         86.85  69.0  105.0      29.12           -4.36             -11.09
(b) Larkin sits: in Carter Mazur for Dylan Larkin
(c) Edvinsson sits: in Albert Johansson for Simon Edvinsson


## 8. The Hellebuyck trade (T1) at opening night

T1 sent Connor Hellebuyck from Winnipeg to New Jersey for Dawson Mercer, Nico Daws, Nick Bjugstad and a prospect (Oct 23). Here it's applied **from opening night**, so the whole season shows the swap.

In [10]:
T1 = SE.run_scenario(B, trade_ids=["T1"], n_sims=N_RUN, seed=SEED)
rows = []
for team in ["WPG", "NJD"]:
    a0, a1 = team_row(BASE, team), team_row(T1, team)
    rows.append({"team": team, "points without T1": a0["points"], "with T1": a1["points"], "change": a1["points"] - a0["points"],
                 "playoff % without": a0["playoff %"], "with": a1["playoff %"], "change ": a1["playoff %"] - a0["playoff %"]})
print(pd.DataFrame(rows).round(2).to_string(index=False))
for team in ["WPG", "NJD"]:
    for nm, res in [("without", BASE), ("with", T1)]:
        g = res["lineups"]
        g = g[(g["team"] == team) & g["dressed"] & (g["group"] == "G")].sort_values("rank")
        print(f"{team} goalies {nm} T1: " + ", ".join(f"{n} ({o:.0f} OVR)" for n, o in zip(g["name"], g["ovr"])))

team  points without T1  with T1  change  playoff % without  with  change 
 WPG              90.31    86.48   -3.83              46.03 36.04    -9.99
 NJD              94.16    98.15    3.99              48.67 59.53    10.86
WPG goalies without T1: Connor Hellebuyck (91 OVR), Stuart Skinner (81 OVR)
WPG goalies with T1: Stuart Skinner (81 OVR), Nico Daws (83 OVR)
NJD goalies without T1: Jake Allen (79 OVR), Nico Daws (83 OVR)
NJD goalies with T1: Connor Hellebuyck (91 OVR), Jake Allen (79 OVR)


## 9. Reality mode: from today, the way things stand

- **Start:** today (the day the schedules were downloaded). Every game before today is locked in with its real result.
- **Trades:** every trade dated on or before today (T1-T6). The dates of T2-T6 are each player's first game with his new team, so on that date he already plays for it.
- **Sit-outs:** Larkin and Edvinsson, who haven't played.

Compared with the opening-night baseline, where the disputes play.

In [11]:
NOW = SE.run_scenario(B, start_date=TODAY, sit_out=[LARKIN, EDVINSSON], n_sims=N_RUN, seed=SEED)
assert (NOW["table"].set_index("team")["real_points"].sort_index() == theirs["points"].sort_index()).all()      # real points locked in correctly
print(f"From {NOW['start_date']}: {NOW['games_locked']} games locked in, {NOW['games_simulated']:,} simulated; "
      f"trades applied: {', '.join(NOW['trades'])}; sitting: Larkin, Edvinsson\n")
d0, d1 = NOW["table"].set_index("team").loc["DET"], BASE["table"].set_index("team").loc["DET"]
r = theirs.loc["DET"]
print(f"Detroit so far: {int(r['gp'])} games, {int(r['wins'])}-{int(r['gp'] - r['wins'] - r['otl'])}-{int(r['otl'])}, {int(r['points'])} points")
print(f"Detroit, reality mode: {d0['points']:.1f} points (10th-90th {d0['p10']:.0f}-{d0['p90']:.0f}), playoffs {d0['playoff_pct']:.1f}%")
print(f"Detroit, opening-night baseline (disputes play): {d1['points']:.1f} points ({d1['p10']:.0f}-{d1['p90']:.0f}), playoffs {d1['playoff_pct']:.1f}%")
print(f"Change: {d0['points'] - d1['points']:+.1f} points, {d0['playoff_pct'] - d1['playoff_pct']:+.1f} playoff points\n")
by_division(NOW["table"], ["team", "real_gp", "real_points", "points", "p10", "p90", "playoff_pct", "division_pct", "presidents_pct"])

From 2026-11-10: 285 games locked in, 1,059 simulated; trades applied: T1, T2, T3, T4, T5, T6; sitting: Larkin, Edvinsson

Detroit so far: 18 games, 7-8-3, 17 points
Detroit, reality mode: 84.5 points (10th-90th 70-100), playoffs 18.0%
Detroit, opening-night baseline (disputes play): 91.2 points (74-109), playoffs 40.2%
Change: -6.7 points, -22.2 playoff points

Eastern, Atlantic
team  real_gp  real_points  points  p10   p90  playoff_pct  division_pct  presidents_pct
 MTL       17           25   107.3 92.0 122.0         84.6          32.4            10.4
 TBL       17           24   104.5 90.0 119.0         79.6          23.4             6.9
 BUF       17           19   102.3 88.0 117.0         74.1          17.8             4.5
 BOS       18           27   101.5 86.0 116.0         70.8          17.0             4.4
 OTT       18           23    92.6 78.0 108.0         40.5           4.6             0.8
 FLA       18           20    90.5 76.0 105.0         33.8           2.8           

## 10. Context check (not a fit): the opening-night baseline vs. the real pace so far

Each team's real points so far, scaled to 84 games, next to the opening-night baseline. **Nothing here is fit to these results.** It's a sanity check, and after about 18 games a team's pace is mostly noise. Notebook 23 put the correlation of opening-night lineup strength with real points percentage at 0.48 after about 15 games.

In [12]:
c = BASE["table"].set_index("team")[["points"]].rename(columns={"points": "baseline"})
c["games"] = theirs["gp"]
c["real points"] = theirs["points"]
c["real pace (per 84)"] = theirs["points"] / theirs["gp"] * 84
c["pace - baseline"] = c["real pace (per 84)"] - c["baseline"]
print(f"Games played so far: {c['games'].min()} to {c['games'].max()}. Correlation of the baseline with the real pace: "
      f"{c['baseline'].corr(c['real pace (per 84)']):.2f}")
print(c.sort_values("pace - baseline").round(1).to_string())

Games played so far: 15 to 20. Correlation of the baseline with the real pace: 0.65
      baseline  games  real points  real pace (per 84)  pace - baseline
team                                                                   
VAN       75.3     20            7                29.4            -45.9
CHI       81.9     18            9                42.0            -39.9
PHI       92.5     20           15                63.0            -29.5
PIT       93.2     18           14                65.3            -27.9
LAK       86.5     15           12                67.2            -19.3
SJS       87.0     20           17                71.4            -15.6
CBJ       90.4     15           14                78.4            -12.0
DET       91.2     18           17                79.3            -11.9
WPG       90.3     18           17                79.3            -11.0
BUF      104.0     17           19                93.9            -10.1
STL       88.5     17           16                79

## 11. Hockey checks

1. **Top and bottom of the baseline:** do they make sense?
2. **Spread:** in one simulated season, how spread out are team points? This is compared with real seasons (2021-22 to 2025-26, scaled to 84 games).
3. **How much of each projection is the team bonus?** The baseline is rerun with the bonus off (same seed). Vancouver and Toronto carry the biggest negative bonuses from 2025-26.

In [13]:
t = BASE["table"].sort_values("points", ascending=False)
print("Top 6:   ", ", ".join(f"{a} {b:.1f}" for a, b in zip(t["team"].head(6), t["points"].head(6))))
print("Bottom 6:", ", ".join(f"{a} {b:.1f}" for a, b in zip(t["team"].tail(6), t["points"].tail(6))))

sim_sd = BASE["points"].std(axis=1).mean()
real_sd = real[real["season"].between(2021, 2025)].groupby("season")["points"].std() * 84 / 82
print(f"\nSpread of team points within one season: simulated {sim_sd:.1f}; real "
      + ", ".join(f"{label(s_)} {v:.1f}" for s_, v in real_sd.items()) + f" (average {real_sd.mean():.1f})")
print(f"Spread of the AVERAGE points across teams: {t['points'].std():.1f} (the averages are smoother than any one season)")
print(f"Best and worst team in a simulated season, on average: {BASE['points'].max(axis=1).mean():.1f} and "
      f"{BASE['points'].min(axis=1).mean():.1f}")

NOB = SE.run_scenario(B, n_sims=N_RUN, seed=SEED, use_bonus=False)
bb = BASE["table"].set_index("team")[["points"]].rename(columns={"points": "with bonus"})
bb["without bonus"] = NOB["table"].set_index("team")["points"]
bb["bonus (points)"] = bb["with bonus"] - bb["without bonus"]
bb["bonus, goals per game"] = pd.Series(B["bonus"])
bb["2025-26 residual (per 84)"] = RESID * 84
print("\nHow much of each team's projection is the bonus (opening night, sorted by the bonus):")
print(bb.sort_values("bonus (points)").round(2).to_string())

Top 6:    COL 109.9, WSH 106.5, DAL 106.0, BUF 104.0, MTL 102.8, CAR 102.5
Bottom 6: TOR 85.1, NYI 83.9, CHI 81.9, SEA 79.9, CGY 76.8, VAN 75.3

Spread of team points within one season: simulated 16.1; real 2021-22 20.8, 2022-23 19.4, 2023-24 18.1, 2024-25 15.2, 2025-26 13.5 (average 17.4)
Spread of the AVERAGE points across teams: 8.7 (the averages are smoother than any one season)
Best and worst team in a simulated season, on average: 125.7 and 58.6

How much of each team's projection is the bonus (opening night, sorted by the bonus):
      with bonus  without bonus  bonus (points)  bonus, goals per game  2025-26 residual (per 84)
team                                                                                             
VAN        75.25          86.96          -11.70                  -0.42                     -33.61
TOR        85.12          95.28          -10.16                  -0.34                     -26.94
CHI        81.86          88.00           -6.13                  

## 12. Same seed, same results, and run times

In [14]:
a = SE.run_scenario(B, start_date=TODAY, n_sims=500, seed=3)
b = SE.run_scenario(B, start_date=TODAY, n_sims=500, seed=3)
assert np.array_equal(a["points"], b["points"]) and a["table"].equals(b["table"])
tl = BASE["tallies"]
n_games = len(SCHED)
assert np.allclose(tl["points"].sum(axis=1), 2 * n_games + tl["otl"].sum(axis=1))     # 2 points a game + 1 per OT/SO game
assert np.allclose(tl["wins"].sum(axis=1), n_games)                                   # one winner per game
print("Same seed, same results: yes. Every simulated season's points add up (2 per game + 1 per OT/SO game): yes")
for n in [1000, 10000]:
    res = SE.run_scenario(B, n_sims=n, seed=11)
    print(f"{n:,} simulated seasons from opening night (1,344 games each, standings and playoffs): {res['seconds']:.1f} seconds")

Same seed, same results: yes. Every simulated season's points add up (2 per game + 1 per OT/SO game): yes
1,000 simulated seasons from opening night (1,344 games each, standings and playoffs): 1.0 seconds
10,000 simulated seasons from opening night (1,344 games each, standings and playoffs): 5.2 seconds


## Takeaways
- **The runner works on the real schedule.**
  - 1,344 games, 84 per team. 285 had been played through Nov 9 when the schedules were downloaded (Nov 10).
  - The results it locks in match the NHL's standings exactly for all 32 teams: points, wins, OT losses, regulation wins, and regulation + OT wins.
- **Notebook 25's step 3 default reproduces:** a_s 5.925, a_g 8.427, home 0.234, beta 0.379, and the bonuses (DET +0.103, VAN -0.422, COL +0.320 goals per game).
- **Goalies:**
  - Teams have 8 to 15 back-to-backs (median 12), so no team needs more than 30 backup starts.
  - Every backup gets 30 starts and every starter's share is 0.643.
- **Opening-night baseline** (no trades, disputes play):
  - **Top:** Colorado 109.9 points (Presidents' Trophy 16.7%), Washington 106.5, Dallas 106.0, Buffalo 104.0, Montreal 102.8, Carolina 102.5.
  - **Bottom:** Vancouver 75.3, Calgary 76.8, Seattle 79.9, Chicago 81.9, the Islanders 83.9, Toronto 85.1.
- **Detroit at opening night:** 91.2 points (10th-90th: 74 to 109), playoffs 40.2%.
  - **Larkin sits:** -3.5 points, -9.0 playoff points. Carter Mazur dresses.
  - **Edvinsson sits:** -0.8 points, -2.0 playoff points. Albert Johansson dresses.
  - **Both sit:** -4.4 points, -11.1 playoff points.
- **The Hellebuyck trade (T1) from opening night:** Winnipeg -3.8 points (-10.0 playoff points), New Jersey +4.0 (+10.9).
  - **A hockey note:** notebook 23's coach score weighs goalies half on past usage. So Winnipeg starts Skinner (81 OVR) ahead of Daws (83) after the trade, and without it New Jersey starts Allen (79) ahead of Daws.
- **Reality mode** (from Nov 10, trades T1-T6, Larkin and Edvinsson sitting):
  - **So far:** Detroit is 7-8-3 (17 points in 18 games).
  - **Projected:** 84.5 points (70 to 100) and an 18.0% playoff chance. That's -6.7 points and -22.2 playoff points vs. the opening-night baseline. The slow start and the two disputes both count.
- **Context check (not a fit):** after 15 to 20 games, the opening-night baseline correlates 0.65 with each team's real points pace.
  - **Biggest gaps below:** Vancouver (7 points in 20 games), Chicago, Philadelphia, Pittsburgh.
  - **Biggest gaps above:** Boston, Carolina, Edmonton, New Jersey, Minnesota.
- **Spread:**
  - **Within one simulated season,** team points spread with an SD of 16.1, vs. 13.5 to 20.8 in real seasons 2021-22 to 2025-26 (average 17.4). A little narrow, but inside the real range.
  - **The best and worst teams** in a simulated season average 125.7 and 58.6 points.
- **The team bonus is a big part of some projections:**
  - **Vancouver** -11.7 points (87.0 without it), **Toronto** -10.2 (95.3), Chicago -6.1, the Rangers -5.8.
  - **Colorado** +9.0, Minnesota +5.8, Carolina +5.5.
  - **Said plainly:** the bonus is notebook 25's shrunk 2025-26 residual, and for Vancouver and Toronto it's the difference between a bubble team and a bottom-five team. Their real starts (Vancouver 7 points in 20 games, Toronto 18 in 19) lean the bonus's way, but that's context, not proof.
- **Teams that look low without a big bonus:** Calgary (76.8) and Seattle (79.9). Their lineups project weak, and the bonus adds or removes little.
- **Speed:** 10,000 seasons in 5.7 seconds, 1,000 in 1.4.
  - The same seed gives the same results.
  - Every simulated season's points add up: 2 per game, plus 1 per OT/SO game.
- **Said plainly:**
  - **Lineups are fixed at the start date,** at full health.
  - **Projections aren't updated in-season,** so a hot or cold start only counts through the points already banked.
  - **The bonus is 2025-26's.**
  - **Skipped tiebreakers:** total wins, head-to-head points, goal difference, goals for.
- **The bundle** for the Simulator page is saved to `data/simulator_2026-27/` (gitignored). The page reads only it and `data/trades_2026-27.csv`.
- **To update results,** set `REFRESH = True` (33 NHL calls) and rerun. Then rerun the app.